# UX Frustration Recognition Experiment

In [ ]:
%matplotlib widget
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import joblib

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

BASE_PATH = "../../experiment-data"
LABELS_SEPARATOR = ","
LABELS_FILENAME = "labels.csv"
FEATURES_SEPARATOR = ";"
FEATURES_DIRECTORY = f"{BASE_PATH}/extracted-features"
FEATURES_FILENAME = f"{FEATURES_DIRECTORY}/all_features.csv"
MODELS_DATE = "20250921"
MODELS_PATH = f"Results/{MODELS_DATE}"
RESULTS_SEPARATOR = ","
RESULTS_DIRECTORY = f"{BASE_PATH}/results"
BIN_LABELS = ["NoStress", "Stress"]
TER_LABELS = ["Relaxed", "Stress", "RealStress"]
QAD_LABELS = ["Relaxed", "Stress", "RealStress", "Amused"]
DEBUG = True

### Import labels and dataset

In [ ]:
labels = pd.read_csv(f"{BASE_PATH}/{LABELS_FILENAME}", sep=LABELS_SEPARATOR, header=0, index_col=0).dropna()
display(labels)

In [ ]:
X = pd.read_csv(f"{FEATURES_FILENAME}", sep=FEATURES_SEPARATOR, header=0, index_col=0)
display(X)
for col in X.columns:
    X[col] = X[col].astype(np.float32)
display(X)


In [ ]:
# Selecting rows that actually have entries in "labels" file
idx = list(X.merge(labels, left_index=True, right_index=True).index)
labels = labels.loc[idx]
x = X.loc[idx]
# Debriefing task has no questionnaire data, it could be used as unseen data and labelled as expected "relax" without ground truth
print(f"Selected {len(x)} entries from X. Not considering {len(X) - len(x)} entries.")

### Helper functions

In [ ]:
def make_results_filename(classification: str, feature_selection: str | None) -> str:
    now = datetime.now()
    timestamp = now.strftime("%Y%m%d-%H%M%S")
    feature_selection = feature_selection if feature_selection is not None else "None"
    return f"{RESULTS_DIRECTORY}/{timestamp}_{classification}_{feature_selection}f"


def show_label_stats(labels: list[str], y: pd.Series):
    display(
        pd.DataFrame(
            {
                "labels": labels,
                "counts": y.value_counts().to_list(),
                "percentage": (y.value_counts(normalize=True) * 100).to_list(),
            }
        )
    )

def show_results(classification: str, feature_selector: str, results: pd.DataFrame, save=False):
    display(results)
    if save:
        res_file = make_results_filename(classification, feature_selector) + ".csv"
        print(f"Saving results to {res_file}")
        results.to_csv(res_file, sep=RESULTS_SEPARATOR)

def show_confusion_matrices(classification: str, feature_selector: str, conf_matrices: dict[str, np.ndarray], labels: list[int], save=False):
    cm_file = make_results_filename(classification, feature_selector) + ".CM.png"
    ncms = len(conf_matrices)
    pairs = [(name, cm) for name, cm in conf_matrices.items()]
    nrows = (ncms // 2) + (ncms % 2)
    fig, axes = plt.subplots(nrows, 2, figsize=(8, nrows * 4))
    for (name, cm), ax in zip(pairs, axes.ravel()):
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        disp.plot(cmap=plt.cm.Blues, ax=ax, colorbar=False)
        ax.set_title(name)
    fig.suptitle("Confusion Matrices")
    plt.tight_layout()
    if save:
        print(f"Saving confusion matrix to {cm_file}")
        plt.savefig(cm_file, dpi=300, format="png")
    plt.show()



### Parameters

In [ ]:
params = {
    "show_confusion_matrices": True,
    "save_confusion_matrices": True,
    "save_results": True,
    "verbose": False,
}

### Classification

In [ ]:
feat_sel = "RFE"
models_paths = {
    "b": {
        "LogisticRegression": f"{MODELS_PATH}/{MODELS_DATE}-182037_binary_LogisticRegression_{feat_sel}f.joblib",
        "MLP": f"{MODELS_PATH}/{MODELS_DATE}-182037_binary_MLPClassifier_{feat_sel}f.joblib",
        "RandomForest": f"{MODELS_PATH}/{MODELS_DATE}-182037_binary_RandomForestClassifier_{feat_sel}f.joblib",
        "SVC": f"{MODELS_PATH}/{MODELS_DATE}-182037_binary_SVC_{feat_sel}f.joblib",
    },
    "t": {
        "LogisticRegression": f"{MODELS_PATH}/{MODELS_DATE}-184421_ternary_LogisticRegression_{feat_sel}f.joblib",
        "MLP": f"{MODELS_PATH}/{MODELS_DATE}-184421_ternary_MLPClassifier_{feat_sel}f.joblib",
        "RandomForest": f"{MODELS_PATH}/{MODELS_DATE}-184421_ternary_RandomForestClassifier_{feat_sel}f.joblib",
        "SVC": f"{MODELS_PATH}/{MODELS_DATE}-184421_ternary_SVC_{feat_sel}f.joblib",
    },
    "q": {
        "LogisticRegression": f"{MODELS_PATH}/{MODELS_DATE}-190744_quaternary_LogisticRegression_{feat_sel}f.joblib",
        "MLP": f"{MODELS_PATH}/{MODELS_DATE}-190744_quaternary_MLPClassifier_{feat_sel}f.joblib",
        "RandomForest": f"{MODELS_PATH}/{MODELS_DATE}-190744s_quaternary_RandomForestClassifier_{feat_sel}f.joblib",
        "SVC": f"{MODELS_PATH}/{MODELS_DATE}-190744_quaternary_SVC_{feat_sel}f.joblib",
    },
}
models: dict[str, dict[str, Pipeline]] = {}
for class_type, paths in models_paths.items():
    models[class_type] = {}
    for model_type, path in paths.items():
        models[class_type][model_type] = joblib.load(path)

for class_type, models in models.items():
    if class_type == "b":
        params["classification"] = "binary"
        params["classes"] = BIN_LABELS
        y = labels["binary-stress"]
    elif class_type == "t":
        params["classification"] = "ternary"
        params["classes"] = TER_LABELS
        y = labels["affect3-class"]
    elif class_type == "q":
        params["classification"] = "quaternary"
        params["classes"] = QAD_LABELS
        y = labels["affect4-class"]

    print(f"\n\n============ {params['classification']} ============\n\n")
    show_label_stats(params["classes"], y)

    df_res = pd.DataFrame(
        {
            "classifier": [],
            "bal-accuracy": [],
            "weighted-f1": [],
            "macro-avg-prec": [],
            "macro-avg-rec": [],
            "macro-avg-f1": [],
            "stress-prec": [],
            "stress-rec": [],
            "stress-f1": [],
        }
    )
    conf_matrices: dict[str, np.ndarray] = {}

    for model_name, model in models.items():
        y_pred = model.predict(x)
        cm = confusion_matrix(y, y_pred)
        report = classification_report(y, y_pred, target_names=params["classes"], output_dict=True)
        conf_matrices[model_name] = cm

        s_report = report["Stress"]
        m_report = report["macro avg"]
        new_row = {
            "classifier": model_name,
            "bal-accuracy": balanced_accuracy_score(y, y_pred),
            "weighted-f1": f1_score(y, y_pred, average="weighted"),
            "macro-avg-prec": m_report["precision"],
            "macro-avg-rec": m_report["recall"],
            "macro-avg-f1": m_report["f1-score"],
            "stress-prec": s_report["precision"],
            "stress-rec": s_report["recall"],
            "stress-f1": s_report["f1-score"],
        }
        df_res.loc[len(df_res)] = new_row

    show_results(params["classification"], feat_sel, df_res, save=params["save_results"])
    if params["show_confusion_matrices"]:
        show_confusion_matrices(params["classification"], feat_sel, conf_matrices, labels=params["classes"], save=params["save_confusion_matrices"])